In [23]:
# !pip install tqdm
#!pip install --upgrade langchain-openai
# !pip install boto3
# !pip install lxml


In [24]:
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

import re
import pandas as pd
import numpy as np
import ast
from xml.dom import minidom

import getpass
import os
from dotenv import load_dotenv
from tqdm import tqdm
import tiktoken
import time
import json
import boto3
from botocore.exceptions import ClientError
import xml.etree.ElementTree as ET
import ast
import xml.etree.ElementTree as ET
from xml.dom import minidom

from lxml import etree
import json
from ast import literal_eval



In [25]:
# Load environment variables from .env file
load_dotenv()

# Set them in the OS environment
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


session = boto3.Session(
    aws_access_key_id='',
    aws_secret_access_key='',
    region_name=''
)



In [26]:
### Getting Original Sentences (Provenances)
all_sentences = pd.read_excel('/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Topic_Sentence_Jodiv01.xlsx')
all_sentences = all_sentences[all_sentences['Model_Topic'] == 'us.deepseek.r1-v1:0']
all_sentences =all_sentences[['Paper_Title','Sentence', 'Provenance',"Text_Original","Category","Section"]]
all_sentences

,Paper_Title,Sentence,Provenance,Text_Original,Category,Section
0,Addressing Unreliability Propagation In Scient...,To identify publications that propagate unreli...,"(339,1)",To identify publications that propagate unreli...,Overall,Related Work
1,Addressing Unreliability Propagation In Scient...,Fu and schneider [45] developed the keystone f...,"(339,2)",Fu and schneider [45] developed the keystone f...,Description,Related Work
2,Addressing Unreliability Propagation In Scient...,Usman and balke [202] developed citation inten...,"(339,3)",Usman and balke [202] developed citation inten...,Description,Related Work
3,Addressing Unreliability Propagation In Scient...,"However, identifying publications that propaga...","(339,4)","However, identifying publications that propaga...",Research gap,Related Work
4,Addressing Unreliability Propagation In Scient...,Some digital library services identify publica...,"(339,5)",Some digital library services identify publica...,Overall,Related Work
...,...,...,...,...,...,...
383,Why Do Some Retracted Articles Continue To Get...,(2) which knowledge claims of the retracted pu...,"(338,46)",(2) which knowledge claims of the rps are the ...,Description,Related Work
384,Why Do Some Retracted Articles Continue To Get...,(3) how do the citing authors describe the cla...,"(338,47)",(3) how do the citing authors describe the cla...,Research gap,Related Work
385,Why Do Some Retracted Articles Continue To Get...,The first two main questions deal with a more ...,"(338,48)",The first two main questions deal with a more ...,Research gap,Related Work
386,Why Do Some Retracted Articles Continue To Get...,The aim of the present work is also to propose...,"(338,49)",The aim of this work is also to propose method...,Extension,Related Work


In [27]:
# Getting Generated Draft
generated_sentences= pd.read_excel('/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Jodi_Draftv01_NoSentence.xlsx')
# generated_sentences= pd.read_excel('/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Jodi_Draftv01.xlsx')
# generated_sentences= generated_sentences[generated_sentences['Model'] == 'us.deepseek.r1-v1:0']
generated_sentences= generated_sentences[generated_sentences['Model'] == 'gpt-4.1-2025-04-14']
generated_sentences

,Thinking,Section,Time,Context Window,Model,Topic,Description
0,NaN,\n1. Citation of Retracted Publications and Ep...,24.39,3531,gpt-4.1-2025-04-14,Citation of Retracted Publications and Epistem...,\n The phenomenon of continued citation o...
1,NaN,\n2. Citation Context Analysis and Citation Ty...,24.25,7534,gpt-4.1-2025-04-14,Citation Context Analysis and Citation Typologies,\n The study and classification of citati...
2,NaN,\n3. Argumentation Theory and Curation in Scie...,9.21,2899,gpt-4.1-2025-04-14,Argumentation Theory and Curation in Scientifi...,\n The application of argumentation theor...


In [28]:
### Get all generated sentences from the Generated Draft.

rows = []
for text in generated_sentences['Section']:
    # Step 1: Extract title (first non-empty line)
    lines = text.strip().splitlines()
    title = next(line.strip() for line in lines if line.strip())

    # Step 2: Extract each <paragraph> block
    paragraphs = re.findall(r'<paragraph>(.*?)</paragraph>', text, re.DOTALL)

    # Step 3: For each paragraph, extract <text> content and split into sentences
    for i, para in enumerate(paragraphs, start=1):
        text_contents = re.findall(r'<text>(.*?)</text>', para, re.DOTALL)
        for content in text_contents:
            sentences = [s.strip() for s in content.split('.') if s.strip()]
            for sentence in sentences:
                # Step 4: Extract all (number,number) patterns and join them with spaces
                provenance_matches = re.findall(r'\(\d+,\d+\)', sentence)
                provenance = ' '.join(provenance_matches) if provenance_matches else None
                
                rows.append({
                    'Title': title,
                    'Paragraph': i,
                    'Generated_Sentence': sentence,
                    'Provenance': provenance
                })

# Step 5: Create DataFrame
df_sentences = pd.DataFrame(rows)

df_sentences['Id_Order'] = range(1, len(df_sentences) + 1)
df_sentences['Has_Provenance'] = np.where(df_sentences['Provenance'].isna(), 'No', 'Yes')
df_sentences

,Title,Paragraph,Generated_Sentence,Provenance,Id_Order,Has_Provenance
0,1. Citation of Retracted Publications and Epis...,1,Research on the citation of retracted publicat...,"(339,1)",1,Yes
1,1. Citation of Retracted Publications and Epis...,1,"In parallel, several citation management tools...","(339,12)",2,Yes
2,1. Citation of Retracted Publications and Epis...,1,Post-retraction citation was first studied in ...,None,3,No
3,1. Citation of Retracted Publications and Epis...,1,g,None,4,No
4,1. Citation of Retracted Publications and Epis...,1,", [212] [41] [10]); the most comprehensive ide...","(8,5)",5,Yes
...,...,...,...,...,...,...
61,3. Argumentation Theory and Curation in Scient...,2,One well-known example of provenance-based cur...,"(45,13)",62,Yes
62,3. Argumentation Theory and Curation in Scient...,2,Argument scheme-based curation approaches rely...,"(45,17) (45,18)",63,Yes
63,3. Argumentation Theory and Curation in Scient...,2,Such argument schemes document the underlying ...,"(45,19)",64,Yes
64,3. Argumentation Theory and Curation in Scient...,3,Argument schemes describe stereotypical reason...,"(45,2)",65,Yes


In [29]:
### Split generated sentences by Provenance
df_sentences_splitted= (
    df_sentences.assign(
        Provenance=df_sentences['Provenance'].str.split()  # Split provenance string by spaces
    )
    .explode('Provenance')  # Create new row for each provenance
    .reset_index(drop=True)  # Reset index after explosion
)
df_sentences_splitted

,Title,Paragraph,Generated_Sentence,Provenance,Id_Order,Has_Provenance
0,1. Citation of Retracted Publications and Epis...,1,Research on the citation of retracted publicat...,"(339,1)",1,Yes
1,1. Citation of Retracted Publications and Epis...,1,"In parallel, several citation management tools...","(339,12)",2,Yes
2,1. Citation of Retracted Publications and Epis...,1,Post-retraction citation was first studied in ...,None,3,No
3,1. Citation of Retracted Publications and Epis...,1,g,None,4,No
4,1. Citation of Retracted Publications and Epis...,1,", [212] [41] [10]); the most comprehensive ide...","(8,5)",5,Yes
...,...,...,...,...,...,...
81,3. Argumentation Theory and Curation in Scient...,2,Argument scheme-based curation approaches rely...,"(45,17)",63,Yes
82,3. Argumentation Theory and Curation in Scient...,2,Argument scheme-based curation approaches rely...,"(45,18)",63,Yes
83,3. Argumentation Theory and Curation in Scient...,2,Such argument schemes document the underlying ...,"(45,19)",64,Yes
84,3. Argumentation Theory and Curation in Scient...,3,Argument schemes describe stereotypical reason...,"(45,2)",65,Yes


In [30]:
### Creating a nested dataframe with the splitted sentences and the original sentences (Provenance)
merged_df = pd.merge(df_sentences_splitted, all_sentences, on='Provenance')

nested_df = ( merged_df.groupby(["Title", "Paragraph","Id_Order","Generated_Sentence"]) .apply(lambda x: x[["Provenance","Paper_Title","Sentence","Text_Original","Category","Section"]].to_dict("records")) .reset_index(name="Source_Details") )

### Convert to a structured DataFrame
final_df = pd.DataFrame({ "Title": nested_df["Title"], "Paragraph": nested_df["Paragraph"],"Id_Order": nested_df["Id_Order"], "Generated_Sentence": nested_df["Generated_Sentence"], "Source_Details": nested_df["Source_Details"] })

/var/folders/40/gnhyngsx3gnc1bc9rjrsgm380000gn/T/ipykernel_6640/1848787023.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nested_df = ( merged_df.groupby(["Title", "Paragraph","Id_Order","Generated_Sentence"]) .apply(lambda x: x[["Provenance","Paper_Title","Sentence","Text_Original","Category","Section"]].to_dict("records")) .reset_index(name="Source_Details") )


In [31]:
final_df['Id_Order_Provenance'] = range(1, len(final_df) + 1)
final_df['Id_Provenance'] = "(Prov."+final_df['Id_Order_Provenance'].astype(str)+")"
final_df

,Title,Paragraph,Id_Order,Generated_Sentence,Source_Details,Id_Order_Provenance,Id_Provenance
0,1. Citation of Retracted Publications and Epis...,1,1,Research on the citation of retracted publicat...,"[{'Provenance': '(339,1)', 'Paper_Title': 'Add...",1,(Prov.1)
1,1. Citation of Retracted Publications and Epis...,1,2,"In parallel, several citation management tools...","[{'Provenance': '(339,12)', 'Paper_Title': 'Ad...",2,(Prov.2)
2,1. Citation of Retracted Publications and Epis...,1,5,", [212] [41] [10]); the most comprehensive ide...","[{'Provenance': '(8,5)', 'Paper_Title': 'Conti...",3,(Prov.3)
3,1. Citation of Retracted Publications and Epis...,2,6,Although a common assumption is that citations...,"[{'Provenance': '(338,5)', 'Paper_Title': 'Why...",4,(Prov.4)
4,1. Citation of Retracted Publications and Epis...,2,7,"However, some scientific papers continue to be...","[{'Provenance': '(338,6)', 'Paper_Title': 'Why...",5,(Prov.5)
5,1. Citation of Retracted Publications and Epis...,2,10,", the example on page 13 of Grieneisen and Zha...","[{'Provenance': '(8,2)', 'Paper_Title': 'Conti...",6,(Prov.6)
6,1. Citation of Retracted Publications and Epis...,2,11,Yet continued citation of retracted papers ret...,"[{'Provenance': '(8,3)', 'Paper_Title': 'Conti...",7,(Prov.7)
7,1. Citation of Retracted Publications and Epis...,3,12,Meta-analyses are an optimal tool to measure t...,"[{'Provenance': '(332,28)', 'Paper_Title': 'Wh...",8,(Prov.8)
8,1. Citation of Retracted Publications and Epis...,3,13,The cumulative nature of knowledge produced by...,"[{'Provenance': '(332,100)', 'Paper_Title': 'W...",9,(Prov.9)
9,1. Citation of Retracted Publications and Epis...,3,14,Meta-analytical methods are protected from the...,"[{'Provenance': '(332,81)', 'Paper_Title': 'Wh...",10,(Prov.10)


In [32]:
# Function to replace trailing (number,number) provenance
def replace_provenance(row):
    sentence = row['Generated_Sentence']
    prov = row['Id_Provenance']
    
    if prov != 'None':
        # Remove all trailing groups like (12,34) with optional spaces
        cleaned = re.sub(r'(\s*\(\d+,\d+\))+$', '', sentence).strip()
        return f"{cleaned} {prov}"
    else:
        return sentence

In [33]:
### Getting the final dataframe of generated sentences with nested attribues and Provenance Id.
final_df = final_df.drop("Id_Order", axis=1)
generated_sentence_final = pd.merge(df_sentences, final_df , on=['Title', 'Paragraph', 'Generated_Sentence'], how='left')
generated_sentence_final['Id_Provenance'] =generated_sentence_final['Id_Provenance'].fillna("None")
# Apply the function
generated_sentence_final['Final_Sentence'] = generated_sentence_final.apply(replace_provenance, axis=1)
generated_sentence_final["Generated_Sentence"] = generated_sentence_final["Generated_Sentence"].str.replace(r"\(\d+,\d+\)", "", regex=True) # It deletes  (#,#) between the text.
generated_sentence_final["Final_Sentence"] = generated_sentence_final["Final_Sentence"].str.replace(r"\(\d+,\d+\)", "", regex=True) # It deletes  (#,#) between the text.
generated_sentence_final

,Title,Paragraph,Generated_Sentence,Provenance,Id_Order,Has_Provenance,Source_Details,Id_Order_Provenance,Id_Provenance,Final_Sentence
0,1. Citation of Retracted Publications and Epis...,1,Research on the citation of retracted publicat...,"(339,1)",1,Yes,"[{'Provenance': '(339,1)', 'Paper_Title': 'Add...",1.0,(Prov.1),Research on the citation of retracted publicat...
1,1. Citation of Retracted Publications and Epis...,1,"In parallel, several citation management tools...","(339,12)",2,Yes,"[{'Provenance': '(339,12)', 'Paper_Title': 'Ad...",2.0,(Prov.2),"In parallel, several citation management tools..."
2,1. Citation of Retracted Publications and Epis...,1,Post-retraction citation was first studied in ...,None,3,No,NaN,NaN,None,Post-retraction citation was first studied in ...
3,1. Citation of Retracted Publications and Epis...,1,g,None,4,No,NaN,NaN,None,g
4,1. Citation of Retracted Publications and Epis...,1,", [212] [41] [10]); the most comprehensive ide...","(8,5)",5,Yes,"[{'Provenance': '(8,5)', 'Paper_Title': 'Conti...",3.0,(Prov.3),", [212] [41] [10]); the most comprehensive ide..."
...,...,...,...,...,...,...,...,...,...,...
61,3. Argumentation Theory and Curation in Scient...,2,One well-known example of provenance-based cur...,"(45,13)",62,Yes,"[{'Provenance': '(45,13)', 'Paper_Title': 'Tow...",50.0,(Prov.50),One well-known example of provenance-based cur...
62,3. Argumentation Theory and Curation in Scient...,2,Argument scheme-based curation approaches rely...,"(45,17) (45,18)",63,Yes,"[{'Provenance': '(45,17)', 'Paper_Title': 'Tow...",51.0,(Prov.51),Argument scheme-based curation approaches rely...
63,3. Argumentation Theory and Curation in Scient...,2,Such argument schemes document the underlying ...,"(45,19)",64,Yes,"[{'Provenance': '(45,19)', 'Paper_Title': 'Tow...",52.0,(Prov.52),Such argument schemes document the underlying ...
64,3. Argumentation Theory and Curation in Scient...,3,Argument schemes describe stereotypical reason...,"(45,2)",65,Yes,"[{'Provenance': '(45,2)', 'Paper_Title': 'Towa...",53.0,(Prov.53),Argument schemes describe stereotypical reason...


In [34]:
def generate_xml_sentence(df):
    """
    Generates XML content in the exact specified format without any wrapping tags.
    Handles both string-encoded and already-parsed Source_Details.
    """
    xml_lines = []
    
    for _, row in df.iterrows():
        # Add generated sentence
        xml_lines.append('<generated_sentence>')
        xml_lines.append(str(row["Generated_Sentence"]))
        xml_lines.append('</generated_sentence>')
        xml_lines.append('\n')


        # xml_lines.append(f'<generated_sentence>{row["Generated_Sentence"]}</generated_sentence>')

        
        xml_lines.append('<source_details>')
        
        try:
            # Handle both string and already-parsed cases
            details = row['Source_Details']
            
            # If it's a string, parse it
            if isinstance(details, str):
                try:
                    details = json.loads(details.replace("'", '"'))
                except json.JSONDecodeError:
                    details = literal_eval(details)
            
            # Ensure we have a list (even if single dict)
            if not isinstance(details, list):
                details = [details]
                
            for detail in details:
                xml_lines.append('   <source>')
                xml_lines.append(f'      <provenance>{detail.get("Provenance", "")}</provenance>')
                xml_lines.append(f'      <text>{detail.get("Sentence", "")}</text>')
                xml_lines.append('   </source>')
                
        except Exception as e:
            print(f"Error processing Source_Details: {e}")
            continue
            
        xml_lines.append('</source_details>')
    
    return '\n'.join(xml_lines)

In [35]:
def extract_think_text(text):
    match = re.search(r"<think>(.*?)</think>(.*)", text, re.DOTALL)
    if match:
        between_think = match.group(1).strip()
        after_think = match.group(2).strip()
    else:
        between_think = ""
        after_think = ""
    
    return between_think, after_think

In [36]:
prompt = f"""
<context>  
You are an expert in text analysis. Your task is to identify the edit_transformation applied to a **generated sentence** based on provided source_details.  
</context>  

_content_

<edit_transformation>
       Copy-Paste with minor edits: Minor edits such as tense/voice change, synonym substitution.
       Paraphrasing: More substantial rewording without altering the meaning. No one-to-one word correspondence, but the message remains the same.
       High-Level Summary: Condenses or generalizes information from multiple source sentences.
       Critique: Adds critique, evaluation, or identifies gaps in the generated sentence. Sometimes introduces information not explicitly present in the source sentence.
</edit_transformation>

<task_requirements>
1. List each provenance and its corresponding edit transformation(s) in the format shown below. A provenance may have more than one transformation.
2. Do not include detailed explanations or examples.
3. Example of valid output:
    - Provenance(10,2): Paraphrasing and High-Level Summary.
    - Provenance(90,4): Copy-Paste with minor edits.
4. Output format:
<response>  
   <transformation_explanation>  
   [Your response here]  
   </transformation_explanation>  
</response> 
</task_requirements>


"""

In [37]:
# def extract_explanation(xml_string):
#     # Parse the XML content
#     root = ET.fromstring(xml_string)
    
#     # Extract the synthesis_explanation text
#     synthesis_explanation = root.find('transformation_explanation').text.strip()

#     return synthesis_explanation


# Regex to strip illegal control characters
_INVALID_XML_CHARS = re.compile(
    u'[\x00-\x08\x0b-\x0c\x0e-\x1f\uFFFE\uFFFF]'
)

def _sanitize_xml(s: str) -> str:
    """Clean up typical bad tokens in model output before parsing."""
    s = s.strip()
    # Remove illegal control chars
    s = re.sub(_INVALID_XML_CHARS, "", s)
    # Escape stray '&' that are not entities
    s = re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#\d+;)', '&amp;', s)
    return s

def extract_explanation(xml_string: str) -> str:
    """Safely parse XML and extract <transformation_explanation> text."""
    s = _sanitize_xml(xml_string)

    try:
        root = ET.fromstring(s)
    except ET.ParseError:
        # If multiple roots or junk outside tags, wrap in <root>
        try:
            root = ET.fromstring(f"<root>{s}</root>")
        except ET.ParseError:
            # If still invalid, just return raw string as fallback
            return s.strip()

    node = root.find('.//transformation_explanation')

    if node is not None and node.text:
        return node.text.strip()
    else:
        # Fallback: return whole text content
        return ET.tostring(root, encoding="unicode", method="text").strip()



In [38]:
def get_explanation_others(df: pd.DataFrame, model_name: str, model_mixed: str):
    model_mixed = model_mixed.upper()

    if model_mixed == "YES" or model_mixed == "NO": 
        start_time = time.time()  # Start timing

        # Initialize the chat model based on the specified model name
        if "gpt" in model_name:
            llm_generator = ChatOpenAI(model_name=model_name, temperature=0)
            if "gpt-4o" in model_name:
                encoding = tiktoken.encoding_for_model(model_name)
            else:
                encoding = tiktoken.get_encoding("cl100k_base")
        else:
            if "o3-mini" in model_name:
                llm_generator = ChatOpenAI(model_name=model_name, temperature=None)
                encoding = tiktoken.get_encoding("cl100k_base")
            else:
                llm_generator = ChatOllama(model=model_name, temperature=0)
                encoding = tiktoken.get_encoding("cl100k_base")

        for i in range(len(df)):
            row_df = df.iloc[[i]]

            if row_df["Has_Provenance"].values[0] == "No":
                df.at[i, 'Response_Transformation'] = "N/A"
                df.at[i, 'Explanation_Transformation'] = "N/A"
                df.at[i, 'Thinking_Transformation'] = "N/A"
                df.at[i, 'Is_Mixed_Model_Transformation'] = "N/A"
                df.at[i, 'Context_Window_Transformation'] = 0
                df.at[i, 'Time_Transformation'] = 0
                df.at[i, 'Model_Transformation'] = model_name
            else:
                # Make sure 'prompt' is defined before using this
                user_question = prompt.replace("_content_", generate_xml_sentence(row_df))

                # Calculate number of tokens
                num_tokens = len(encoding.encode(user_question))
                print(f"\n Model Name: {model_name} - Context Window: {num_tokens} tokens \n")
                print(user_question)

                # Message formatting based on model type
                if "gpt" in model_name or "o3-mini" in model_name:
                    messages = [
                        {"role": "system", "content": "You are an expert in text analysis. Your task is to identify the edit_transformation applied to a **generated sentence** based on provided source_details."},
                        {"role": "user", "content": user_question}
                    ]
                else:
                    template = ChatPromptTemplate.from_messages([
                        ("system", "You are an expert in text analysis. Your task is to identify the edit_transformation applied to a **generated sentence** based on provided source_details."),
                        ("human", "{user_input}")
                    ])
                    messages = template.format_messages(user_input=user_question)

                # Invoke model
                response = llm_generator.invoke(messages)
                response_temp = response.content

                # Separate thinking process and final answer
                if "gpt" in model_name or "o3-mini" in model_name:
                    thinking = "NA"
                    response_final = response_temp  
                else:
                    thinking, response_final = extract_think_text(response_temp)

                end_time = time.time()
                elapsed_time = round(end_time - start_time, 2)

                print(f"\nTotal execution time: {elapsed_time} seconds")
                print("Response: \n", response_final)

                # Assign response values to current row
                df.at[i, 'Response_Transformation'] = response_final
                df.at[i, 'Explanation_Transformation'] = extract_explanation(response_final)
                df.at[i, 'Thinking_Transformation'] = thinking
                df.at[i, 'Is_Mixed_Model_Transformation'] = model_mixed
                df.at[i, 'Context_Window_Transformation'] = num_tokens
                df.at[i, 'Time_Transformation'] = elapsed_time
                df.at[i, 'Model_Transformation'] = model_name

        return df

    else:
        print("The mixed model parameter should be YES or NO and you have assigned: ", model_mixed)
        return None


In [39]:
def get_explanation_bedrock(df: pd.DataFrame, model_name: str, model_mixed: str):
    model_mixed = model_mixed.upper()

    if model_mixed == "YES" or model_mixed == "NO":
        start_time = time.time()
        encoding = tiktoken.get_encoding("cl100k_base")

        # Create a Bedrock runtime client
        bedrock = session.client(service_name='bedrock-runtime')

        # System prompt
        system_prompt = "You are an expert in text analysis. Your task is to identify the edit_transformation applied to a **generated sentence** based on provided source_details."
        system_messages = [{"text": system_prompt}]

        for i in range(len(df)):
            row_df = df.iloc[[i]]
            if row_df["Has_Provenance"].values[0] == "No":
                df.at[i,'Response_Transformation']="N/A"
                df.at[i,'Explanation_Transformation']="N/A"
                df.at[i,"Thinking_Transformation"] = "N/A"
                df.at[i,"Is_Mixed_Model_Transformation"] = "N/A"
                df.at[i,"Context_Window_Transformation"] = 0
                df.at[i,"Time_Transformation"] = 0
                df.at[i,"Model_Transformation"] = model_name
            else:

                if "mistral-large" in model_name:
                    user_prompt = prompt_mistral.replace("_content_", generate_xml_sentence(row_df))
                else:
                    user_prompt = prompt.replace("_content_", generate_xml_sentence(row_df))

                num_tokens = len(encoding.encode(user_prompt))
                print(f"\n Model Name: {model_name} - Context Window: {num_tokens} tokens \n")
                print(user_prompt)

                user_messages = [{"role": "user", "content": [{"text": user_prompt}]}]

                thinking = None
                response_final = None

                try:
                    if "claude-3-7-sonnet" in model_name and model_mixed == "YES":
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 5000, # Max 131072
                                "temperature": 1,
                            },
                            additionalModelRequestFields={
                                'thinking': {
                                    "type": "enabled",
                                    "budget_tokens": 2000
                                }
                            }
                        )
                    elif "claude-3-7-sonnet" in model_name and model_mixed == "NO":
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 5000, # Max 131072
                                "temperature": 0,
                                "topP": 0
                            }
                        )
                    elif "deepseek.r1" in model_name:
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 32000, # Max 32768
                                "temperature": 0,
                                "topP": 0.0
                            },
                        )
                    elif "mistral-large" in model_name:
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 8000, # Max 8000
                                "temperature": 0,
                                "topP": 0.0
                            },
                        )
                    elif "llama3-3" in model_name:
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 128000,
                                "temperature": 0,
                                "topP": 0.0
                            },
                        )

                    # Extract content from response
                    contents = response["output"]["message"]["content"]

                    for content in contents:
                        if "reasoningContent" in content:
                            thinking = content["reasoningContent"]["reasoningText"]["text"]
                        if "text" in content:
                            response_final = content["text"]

                    end_time = time.time()
                    elapsed_time = round(end_time - start_time, 2)
                    print(f"\nTotal execution time: {elapsed_time} seconds")
                    print("Response: \n", response_final)

                    df.at[i,'Response_Transformation']=response_final
                    df.at[i,'Explanation_Transformation']=extract_explanation(response_final)
                    df.at[i,"Thinking_Transformation"] = thinking
                    df.at[i,"Is_Mixed_Model_Transformation"] = model_mixed
                    df.at[i,"Context_Window_Transformation"] = num_tokens
                    df.at[i,"Time_Transformation"] = elapsed_time
                    df.at[i,"Model_Transformation"] = model_name

                except Exception as e:
                    print(f"An error occurred: {str(e)}")
                    continue

        return df

    else:
        print("The mixed model parameter should be YES or NO and you have assigned: ", model_mixed)
        return None

In [40]:
def get_explanation(df: pd.DataFrame, model_name: str, model_mixed: str):
    
    if "deepseek.r1" in model_name or "anthropic.claude-3-7-sonnet" in model_name or "mistral-large" in model_name or "llama3-3" in model_name:
        response_df=get_explanation_bedrock(df,model_name,model_mixed)  
    else:
        response_df=get_explanation_others(df,model_name,model_mixed)
    
    return response_df

In [41]:
# generated_sentence_final1=generated_sentence_final.head(5)
# generated_sentence_final1

In [42]:
# models = ["gpt-4.1-2025-04-14","us.deepseek.r1-v1:0"]
# models_mixed=["No","No"]

models = ["gpt-4.1-2025-04-14"]
# models = ["us.deepseek.r1-v1:0"]
models_mixed=["No"]

all_sentences_explanation = pd.DataFrame()

# Loop through the list and print each name
for model,model_mixed in zip(models,models_mixed):
    
    sentences_explanation= get_explanation(generated_sentence_final, model,model_mixed)
    all_sentences_explanation= pd.concat([all_sentences_explanation, sentences_explanation], ignore_index=True)


 Model Name: gpt-4.1-2025-04-14 - Context Window: 402 tokens 


<context>  
You are an expert in text analysis. Your task is to identify the edit_transformation applied to a **generated sentence** based on provided source_details.  
</context>  

<generated_sentence>
Research on the citation of retracted publications has focused on understanding the mechanisms by which unreliable work continues to influence the scientific literature, with approaches developed for auditing literature that cites unreliable work, based on the recognition that not all citations are equally influential in citing publications [22] 
</generated_sentence>


<source_details>
   <source>
      <provenance>(339,1)</provenance>
      <text>To identify publications that propagate unreliability, researchers have developed approaches for auditing literature that cites unreliable work, relying on the fact that not all citations are equally influential in citing publications [22].</text>
   </source>
</source_details>

In [43]:
all_sentences_explanation 

,Title,Paragraph,Generated_Sentence,Provenance,Id_Order,Has_Provenance,Source_Details,Id_Order_Provenance,Id_Provenance,Final_Sentence,Response_Transformation,Explanation_Transformation,Thinking_Transformation,Is_Mixed_Model_Transformation,Context_Window_Transformation,Time_Transformation,Model_Transformation
0,1. Citation of Retracted Publications and Epis...,1,Research on the citation of retracted publicat...,"(339,1)",1,Yes,"[{'Provenance': '(339,1)', 'Paper_Title': 'Add...",1.0,(Prov.1),Research on the citation of retracted publicat...,<response> \n <transformation_explanation> ...,"Provenance(339,1): Paraphrasing and Copy-Paste...",NA,NO,402.0,1.41,gpt-4.1-2025-04-14
1,1. Citation of Retracted Publications and Epis...,1,"In parallel, several citation management tools...","(339,12)",2,Yes,"[{'Provenance': '(339,12)', 'Paper_Title': 'Ad...",2.0,(Prov.2),"In parallel, several citation management tools...",<response> \n <transformation_explanation> ...,"Provenance(339,12): Copy-Paste with minor edits.",NA,NO,368.0,3.20,gpt-4.1-2025-04-14
2,1. Citation of Retracted Publications and Epis...,1,Post-retraction citation was first studied in ...,None,3,No,NaN,NaN,None,Post-retraction citation was first studied in ...,N/A,N/A,N/A,N/A,0.0,0.00,gpt-4.1-2025-04-14
3,1. Citation of Retracted Publications and Epis...,1,g,None,4,No,NaN,NaN,None,g,N/A,N/A,N/A,N/A,0.0,0.00,gpt-4.1-2025-04-14
4,1. Citation of Retracted Publications and Epis...,1,", [212] [41] [10]); the most comprehensive ide...","(8,5)",5,Yes,"[{'Provenance': '(8,5)', 'Paper_Title': 'Conti...",3.0,(Prov.3),", [212] [41] [10]); the most comprehensive ide...",<response> \n <transformation_explanation> ...,"- Provenance(8,5): Copy-Paste with minor edits.",NA,NO,468.0,4.55,gpt-4.1-2025-04-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,3. Argumentation Theory and Curation in Scient...,2,One well-known example of provenance-based cur...,"(45,13)",62,Yes,"[{'Provenance': '(45,13)', 'Paper_Title': 'Tow...",50.0,(Prov.50),One well-known example of provenance-based cur...,<response> \n <transformation_explanation> ...,"Provenance(45,13): Copy-Paste with minor edits.",NA,NO,384.0,57.75,gpt-4.1-2025-04-14
62,3. Argumentation Theory and Curation in Scient...,2,Argument scheme-based curation approaches rely...,"(45,17) (45,18)",63,Yes,"[{'Provenance': '(45,17)', 'Paper_Title': 'Tow...",51.0,(Prov.51),Argument scheme-based curation approaches rely...,<response> \n <transformation_explanation> ...,"Provenance(45,17): Copy-Paste with minor edits...",NA,NO,444.0,58.93,gpt-4.1-2025-04-14
63,3. Argumentation Theory and Curation in Scient...,2,Such argument schemes document the underlying ...,"(45,19)",64,Yes,"[{'Provenance': '(45,19)', 'Paper_Title': 'Tow...",52.0,(Prov.52),Such argument schemes document the underlying ...,<response> \n <transformation_explanation> ...,"Provenance(45,19): Copy-Paste with minor edits.",NA,NO,376.0,59.83,gpt-4.1-2025-04-14
64,3. Argumentation Theory and Curation in Scient...,3,Argument schemes describe stereotypical reason...,"(45,2)",65,Yes,"[{'Provenance': '(45,2)', 'Paper_Title': 'Towa...",53.0,(Prov.53),Argument schemes describe stereotypical reason...,<response> \n <transformation_explanation> ...,"Provenance(45,2): Copy-Paste with minor edits.",NA,NO,350.0,61.26,gpt-4.1-2025-04-14


In [44]:
path='/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Jodi_Draft_TransformationEditsv01_NoSentence.xlsx'
# path='/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Jodi_Draft_TransformationEditsv01_Sentence.xlsx'

all_sentences_explanation .to_excel(path, index=False, engine='openpyxl')